# superGroups · controlled experiments
These four approaches were developed sequentially on a T4: duration transitions → response-dependent data → relative-pitch listening → explicit onset-pitch supervision.

Choose **Runtime → Change runtime type → T4 GPU**. Select a run below; each run trains for 1,000 updates. Review the results before spending more compute. This reproduces one experiment, not all four automatically. Data is synthetic, and exact-response scores are not a general musical-quality measure.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, torch
assert torch.cuda.is_available(), 'Choose a GPU runtime'
print(torch.cuda.get_device_name(0), torch.__version__)
REPO = Path('/content/superGroups-reproduction')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/nicjams/nicjams.github.io.git',str(REPO)],check=True)
# Pin the code used for the fourth experiment; earlier options remain compatible.
subprocess.run(['git','checkout','3ddb78c'],cwd=REPO,check=True)
os.chdir(REPO/'superGroups')
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
def logged(command):
    with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True) as proc:
        for line in proc.stdout: print(line,end='',flush=True)
        if proc.wait(): raise RuntimeError('Command failed')
def run(*args): logged([sys.executable,'-m','supergroups.cli',*map(str,args)])


In [ ]:
RUN_ID = 4  # 1=durations, 2=response data, 3=pitch listener, 4=pitch objective
assert RUN_ID in (1,2,3,4)
ROOT = Path(f'/content/superGroups-run{RUN_ID}')
ROOT.mkdir(exist_ok=True)
DATA = ROOT/'dataset.npz'
RUN = ROOT/'checkpoint'
run('demo-data' if RUN_ID == 1 else 'conversation-data','--out',DATA,'--songs',256,'--steps',128)
flags = ['--local-transition','--conditional-weights']
if RUN_ID >= 3: flags += ['--pitch-context']
if RUN_ID >= 4: flags += ['--pitch-loss',.2]
run('train','--data',DATA,'--out',RUN,'--updates',1000,'--eval-every',200,'--device','cuda',*flags)


## Evaluate before deciding on the next run
The state-prediction loss measures held-out data fit. Context removal and shuffled context reveal dependence on other players. For runs 2–4, reply-pitch tests compare the predicted response to a known cue, both with correct historical notes and with generated history. Exogenous keyboard cue roots are intentionally unpredictable. Evaluate bass/lead responses separately.


In [ ]:
logged([sys.executable,'-m','supergroups.evaluate',str(RUN/'best.pt'),str(DATA),'--out',str(ROOT/'eval'),'--device','cuda'])
if RUN_ID >= 2:
    logged([sys.executable,'-m','supergroups.response_probe',str(RUN/'best.pt'),str(DATA),'--out',str(ROOT/'probe'),'--device','cuda'])
from IPython.display import Audio, display
print('Fully generated ensemble:')
display(Audio(filename=str(ROOT/'eval/listening-17.wav')))
if RUN_ID >= 2:
    print('Responding to a fixed keyboard performance:')
    display(Audio(filename=str(ROOT/'probe/anchored.wav')))


## Save before disconnecting
Download the dataset, checkpoint, metrics, MIDI and audio. Runtime storage is temporary. Checkpoints include optimizer/scaler/RNG state, so `best.pt` can also be used with `--resume`. Keep the selected experiment flags when resuming, especially `--pitch-loss 0.2` for run 4.


In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive(f'/content/superGroups-run{RUN_ID}-results','zip',ROOT)
files.download(archive)
